# Pandas Data Aggregation

## Joining dataframes

In [2]:
import pandas as pd

In [3]:
## Joining
staff_df = pd.DataFrame([{'Name': 'Kelly', 'Role': 'Director of HR'},
                         {'Name': 'Sally', 'Role': 'Course liasion'},
                         {'Name': 'James', 'Role': 'Grader'}])
staff_df = staff_df.set_index('Name')

student_df = pd.DataFrame([{'Name': 'James', 'School': 'Business'},
                           {'Name': 'Mike', 'School': 'Law'},
                           {'Name': 'Sally', 'School': 'Engineering'}])
student_df = student_df.set_index('Name')

print(staff_df.head())
print(student_df.head())

                 Role
Name                 
Kelly  Director of HR
Sally  Course liasion
James          Grader
            School
Name              
James     Business
Mike           Law
Sally  Engineering


In [4]:
# outer join by index
df1 = pd.merge(staff_df, student_df, how='outer', left_index=True, right_index=True)
df1

,Role,School
Name,,
James,Grader,Business
Kelly,Director of HR,NaN
Mike,NaN,Law
Sally,Course liasion,Engineering


In [5]:
# inner, left, and right joins by index
df2 = pd.merge(staff_df, student_df, how='inner', left_index=True, right_index=True)
df3 = pd.merge(staff_df, student_df, how='left', left_index=True, right_index=True)
df4 = pd.merge(staff_df, student_df, how='right', left_index=True, right_index=True)

In [7]:
# join using columns (not index), this is the more common way
staff_df = staff_df.reset_index()
student_df = student_df.reset_index()
pd.merge(staff_df, student_df, how='right', on='Name')

,index_x,Name,Role,index_y,School
0,2.0,James,Grader,0,Business
1,NaN,Mike,NaN,1,Law
2,1.0,Sally,Course liasion,2,Engineering


If there are conflicts between the tables (i.e., same column names), Pandas will assign _x and _y suffices to the left and right dataframes.

In [8]:
# Join on two columns
staff_df = pd.DataFrame([{'First Name': 'Kelly', 'Last Name': 'Desjardins',
                          'Role': 'Director of HR'},
                         {'First Name': 'Sally', 'Last Name': 'Brooks',
                          'Role': 'Course liasion'},
                         {'First Name': 'James', 'Last Name': 'Wilde',
                          'Role': 'Grader'}])
student_df = pd.DataFrame([{'First Name': 'James', 'Last Name': 'Hammond',
                            'School': 'Business'},
                           {'First Name': 'Mike', 'Last Name': 'Smith',
                            'School': 'Law'},
                           {'First Name': 'Sally', 'Last Name': 'Brooks',
                            'School': 'Engineering'}])

pd.merge(staff_df, student_df, how='inner', on=['First Name','Last Name'])

,First Name,Last Name,Role,School
0,Sally,Brooks,Course liasion,Engineering


## Stacking dataframes

In [11]:
# suppose we have df1, df2, df3
stacked_df = pd.concat([df1, df2, df3])

# If we want to add a column to identify which observation is from which dataframe:
stacked_df = pd.concat([df1, df2, df3], keys=['2011','2012','2013'])

## Grouping Data

The idea behind the groupby() function is  that it takes some dataframe, splits it into chunks based on some key values, applies computation on those  chunks, then combines the results back together into another dataframe. In pandas this is referred to as the split-apply-combine pattern.

In [12]:
import pandas as pd
import numpy as np

### Grouping by column unique values

In [28]:
# import US census data, keep only county-level sum level
df = pd.read_csv('data_intro/census.csv')
df = df[df['SUMLEV']==50]
df.head()

,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861
5,50,3,6,1,9,Alabama,Blount County,57322,57322,57373,...,1.807375,-1.177622,-1.748766,-2.062535,-1.369970,1.859511,-0.848580,-1.402476,-1.577232,-0.884411


In [21]:
# the df.groupby('STNAME') object is iterable with the group name (the unique values in 'STNAME') and the dataframe for each 'STNAME'.
for group, frame in df.groupby('STNAME'):
    avg = np.average(frame['CENSUS2010POP'])
    print('Counties in state ' + group +
          ' have an average population of ' + str(avg))

Counties in state Alabama have an average population of 71339.34328358209
Counties in state Alaska have an average population of 24490.724137931036
Counties in state Arizona have an average population of 426134.4666666667
Counties in state Arkansas have an average population of 38878.90666666667
Counties in state California have an average population of 642309.5862068966
Counties in state Colorado have an average population of 78581.1875
Counties in state Connecticut have an average population of 446762.125
Counties in state Delaware have an average population of 299311.3333333333
Counties in state District of Columbia have an average population of 601723.0
Counties in state Florida have an average population of 280616.5671641791
Counties in state Georgia have an average population of 60928.63522012578
Counties in state Hawaii have an average population of 272060.2
Counties in state Idaho have an average population of 35626.86363636364
Counties in state Illinois have an average populat

### Grouping by function

In [29]:
# Write a function that returns some values (e.g., 0, 1, or 2) based on a column
# First, set the column that we want to apply the function as index
df1 = df.set_index('STNAME')

# Write a function that returns 1 if state name starts with a letter smaller than M, etc.
def set_batch_number(x):
    if x[0] < 'M':
        return 0
    if x[0] < 'Q':
        return 1
    return 2

# Split the df using the function, the function applies by default to the index
for group, frame in df1.groupby(set_batch_number):
    print(f"There are {str(len(frame))} records in group {str(group)} for processing.")

There are 1177 records in group 0 for processing.
There are 1134 records in group 1 for processing.
There are 831 records in group 2 for processing.


In [30]:
# If not setting index
for group, frame in df.groupby(df['STNAME'].apply(set_batch_number)):
    print(f"There are {str(len(frame))} records in group {str(group)} for processing.")

There are 1177 records in group 0 for processing.
There are 1134 records in group 1 for processing.
There are 831 records in group 2 for processing.


### Group by multiple columns

In [35]:
df=pd.read_csv("data_intro/listings.csv")

In [32]:
# Group by index
df=df.set_index(["cancellation_policy","review_scores_value"])
for group, frame in df.groupby(level=(0,1)):
    print(group)


('flexible', 2.0)
('flexible', 4.0)
('flexible', 5.0)
('flexible', 6.0)
('flexible', 7.0)
('flexible', 8.0)
('flexible', 9.0)
('flexible', 10.0)
('moderate', 2.0)
('moderate', 4.0)
('moderate', 6.0)
('moderate', 7.0)
('moderate', 8.0)
('moderate', 9.0)
('moderate', 10.0)
('strict', 2.0)
('strict', 3.0)
('strict', 4.0)
('strict', 5.0)
('strict', 6.0)
('strict', 7.0)
('strict', 8.0)
('strict', 9.0)
('strict', 10.0)
('super_strict_30', 6.0)
('super_strict_30', 7.0)
('super_strict_30', 8.0)
('super_strict_30', 9.0)
('super_strict_30', 10.0)


In [34]:
# Use a function to modify the group
def grouping_fun(item):
    # Check the "review_scores_value" portion of the index. item is in the format of
    # (cancellation_policy,review_scores_value)
    if item[1] == 10.0:
        return (item[0],"10.0")
    else:
        return (item[0],"not 10.0")

# No that the function tranforms the initial group to a more compact group
for group, frame in df.groupby(by=grouping_fun):
    print(group)

('flexible', '10.0')
('flexible', 'not 10.0')
('moderate', '10.0')
('moderate', 'not 10.0')
('strict', '10.0')
('strict', 'not 10.0')
('super_strict_30', '10.0')
('super_strict_30', 'not 10.0')


## Aggregation after Grouping

In [37]:
# load airbnb listings data, reset index, and apply np.average function after grouping
df = pd.read_csv("data_intro/listings.csv")
df = df.reset_index()
df.groupby("cancellation_policy").agg({"review_scores_value":np.average})

,review_scores_value
cancellation_policy,
flexible,NaN
moderate,NaN
strict,NaN
super_strict_30,NaN


In [39]:
# The above method has a NaN problem, as np.average doesn't ignore NaNs
# Instead, we can use np.nanmean, which ignores NaNs.
df.groupby("cancellation_policy").agg({"review_scores_value":np.nanmean})

,review_scores_value
cancellation_policy,
flexible,9.237421
moderate,9.307398
strict,9.081441
super_strict_30,8.537313


In [40]:
# Use a dictionary to apply group-level calculation for multiple columns
df.groupby("cancellation_policy").agg({"review_scores_value":(np.nanmean,np.nanstd),
                                       "reviews_per_month":np.nanmean})

review_scores_value           reviews_per_month
                                nanmean    nanstd           nanmean
cancellation_policy                                                
flexible                       9.237421  1.095409          1.829210
moderate                       9.307398  0.859311          2.391922
strict                         9.081441  1.040123          1.873467
super_strict_30                8.537313  0.834487          0.340143

## Transformation after grouping

Transformation is different from aggregation. Where agg() returns a single value per column, so one row per group, tranform() returns an object that is the same size as the group. Essentially, it broadcasts the function you supply over the grouped dataframe, returning a new dataframe. This makes combining data later easy.

After obtaining the tranformed dataframe, merge it back to the original dataframe so you get group averages.

In [49]:
# For instance, suppose we want to include the average rating values in a given group by cancellation policy,
# but preserve the dataframe shape so that we could generate a difference between an individual observation
# and the sum.
df = pd.read_csv("data_intro/listings.csv")

# First, lets define just some subset of columns we are interested in
cols=['cancellation_policy','review_scores_value']
# Now lets transform it, I'll store this in its own dataframe
transform_df=df[cols].groupby('cancellation_policy').transform(np.nanmean)
print(transform_df.shape)
print(type(transform_df))
print(transform_df.head())

(3585, 1)
<class 'pandas.DataFrame'>
   review_scores_value
0             9.307398
1             9.307398
2             9.307398
3             9.307398
4             9.237421


In [50]:
# Merge the transformed df back to the original df
transform_df.rename({'review_scores_value':'mean_review_scores'}, axis='columns', inplace=True)
df=df.merge(transform_df, left_index=True, right_index=True)
df.head()

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month,mean_review_scores
0,12147973,https://www.airbnb.com/rooms/12147973,20160906204935,2016-09-07,Sunny Bungalow in the City,"Cozy, sunny, family home. Master bedroom high...",The house has an open and cozy feel at the sam...,"Cozy, sunny, family home. Master bedroom high...",none,"Roslindale is quiet, convenient and friendly. ...",...,f,NaN,NaN,f,moderate,f,f,1,NaN,9.307398
1,3075044,https://www.airbnb.com/rooms/3075044,20160906204935,2016-09-07,Charming room in pet friendly apt,Charming and quiet room in a second floor 1910...,Small but cozy and quite room with a full size...,Charming and quiet room in a second floor 1910...,none,"The room is in Roslindale, a diverse and prima...",...,f,NaN,NaN,t,moderate,f,f,1,1.30,9.307398
2,6976,https://www.airbnb.com/rooms/6976,20160906204935,2016-09-07,Mexican Folk Art Haven in Boston,"Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...",none,The LOCATION: Roslindale is a safe and diverse...,...,f,NaN,NaN,f,moderate,t,f,1,0.47,9.307398
3,1436513,https://www.airbnb.com/rooms/1436513,20160906204935,2016-09-07,Spacious Sunny Bedroom Suite in Historic Home,Come experience the comforts of home away from...,Most places you find in Boston are small howev...,Come experience the comforts of home away from...,none,Roslindale is a lovely little neighborhood loc...,...,f,NaN,NaN,f,moderate,f,f,1,1.00,9.307398
4,7651065,https://www.airbnb.com/rooms/7651065,20160906204935,2016-09-07,Come Home to Boston,"My comfy, clean and relaxing home is one block...","Clean, attractive, private room, one block fro...","My comfy, clean and relaxing home is one block...",none,"I love the proximity to downtown, the neighbor...",...,f,NaN,NaN,f,flexible,f,f,1,2.25,9.237421


In [52]:
# Now we can create new columns like absolute deviation from the means by group
df['mean_diff']=np.absolute(df['review_scores_value']-df['mean_review_scores'])
df[['mean_diff', 'review_scores_value', 'mean_review_scores']].head()

,mean_diff,review_scores_value,mean_review_scores
0,NaN,NaN,9.307398
1,0.307398,9.0,9.307398
2,0.692602,10.0,9.307398
3,0.692602,10.0,9.307398
4,0.762579,10.0,9.237421


## Filtering and Applying after Grouping

In [57]:
# Filter by passing in lambda functions
df_filtered = df.groupby('cancellation_policy').filter(lambda x: np.nanmean(x['review_scores_value'])>9.2)
df_filtered.head()[['cancellation_policy', 'review_scores_value']]

,cancellation_policy,review_scores_value
0,moderate,NaN
1,moderate,9.0
2,moderate,10.0
3,moderate,10.0
4,flexible,10.0


In [69]:
# Apply function to the group project
df=pd.read_csv("data_intro/listings.csv")
df=df[['cancellation_policy','review_scores_value']]

def calc_mean_review_scores(group):
    # group is a dataframe just of whatever we have grouped by, e.g. cancellation policy, so we can treat
    # this as the complete dataframe
    avg=np.nanmean(group["review_scores_value"])
    # now broadcast our formula and create a new column
    group['review_scores_mean'] = avg
    group["review_scores_mean_deviation"]=np.abs(avg-group["review_scores_value"])
    return group

# Now just apply this to the groups
df1 = df.groupby('cancellation_policy').apply(calc_mean_review_scores)
df1.sample(10)

review_scores_value  review_scores_mean  \
cancellation_policy                                                 
super_strict_30     2135                  8.0            8.537313   
flexible            815                   8.0            9.237421   
moderate            289                  10.0            9.307398   
strict              818                   8.0            9.081441   
moderate            452                  10.0            9.307398   
strict              2497                 10.0            9.081441   
                    1703                  9.0            9.081441   
                    802                   NaN            9.081441   
                    1592                  9.0            9.081441   
                    733                  10.0            9.081441   

                          review_scores_mean_deviation  
cancellation_policy                                     
super_strict_30     2135                      0.537313  
flexible            815                       1.237421  
moderate            289                       0.692602  
strict              818                       1.081441  
moderate            452                       0.692602  
strict              2497                      0.918559  
                    1703                      0.081441  
                    802                            NaN  
                    1592                      0.081441  
                    733                       0.918559